# FREUID Challenge — v8: Noise/Sharpness Equalization Fine-Tune

**Diagnosis chain so far:** face-swap shortcut (v3 Grad-CAM) -> fixed via face occlusion (v4,
LB 0.279 -> 0.221) -> `is_digital` ruled out as a further cause (LB 0.221 -> 0.219, noise-level)
-> Grad-CAM on v4 showed a document-*format* split (ID-card vs driving-license) -> confirmed via
pixel statistics: `noise_sigma` and `laplacian_var` differ sharply between `MAURITIUS/ID` and the
DL-format types, in BOTH genuine and fraud images (label-independent), with `jpeg_qmean` identical
across all types (ruling out compression-quality as the cause). This points at a rendering/capture
pipeline fingerprint the model can use as a shortcut instead of genuine forgery cues.

**Strategy: warm-start fine-tune, not from-scratch retrain.** v4's checkpoints are already
essentially converged (fold0 OOF FREUID 0.0003 at epoch 17, fold1 0.0007 at epoch 8) — a short,
low-LR fine-tune phase with a NEW augmentation that actively equalizes noise/sharpness toward a
common mid-band should be enough to break the shortcut without needing to relearn everything from
random init. This is faster and lower-risk than a full retrain: less new code running for less
wall-clock time, on top of weights already known to work.

**Important ceiling-effect caveat:** OOF FREUID is already near 0 on the SAME validation fold used
before. A fine-tune that successfully removes a shortcut the model was previously exploiting on
that same in-distribution val data may not, and does not need to, beat the old OOF number to be a
real improvement on the actual (out-of-distribution-ish) test set. **The real signal here is the
next LB submission, not the OOF metric.** Every epoch is saved unconditionally for exactly this
reason — "only save on new best" would likely never fire again given how close to 0 the old best
already is.

**Safety features baked in, given this needs to survive a 5-6hr unattended run:**
- Fail-fast GPU compatibility check (catches a P100-vs-T4 accelerator mismatch immediately)
- Face-bbox cache reuse is **mandatory** — recomputing it cost ~2.1hrs in v4; this notebook refuses
  to silently eat that time again and fails fast with instructions if the cache isn't attached
- v4 checkpoints are loaded from a **separate, read-only-in-spirit source directory** and never
  overwritten — a guaranteed fallback to the current 0.22-scoring submission always exists
- Every fine-tune epoch is checkpointed unconditionally (not just on improvement) for crash-safety
- A hard wall-clock budget (`MAX_WALLCLOCK_HOURS`) cuts each fold's training short gracefully if it
  runs long, rather than risking the whole run
- Per-fold `try/except` isolation — one fold failing doesn't take down the other or the final
  submission generation
- Final submission generation always runs, using whichever per-fold checkpoint is best-available
  (v5 fine-tuned if it succeeded, v4 original as automatic fallback if not) — you always get a
  valid, submittable file out of this notebook regardless of what goes wrong mid-run


## 1. Environment Setup

In [ ]:
import os
import time
import math
import json
import random
import shutil
import warnings
import traceback
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

os.environ['NO_ALBUMENTATIONS_UPDATE'] = '1'
os.environ['HF_HUB_OFFLINE']           = '1'
os.environ['TRANSFORMERS_OFFLINE']     = '1'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats
import scipy.optimize
import cv2
import pickle
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.isotonic import IsotonicRegression

warnings.filterwarnings('ignore')

SEED = 42

def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

seed_everything(SEED)

FORCE_CPU = False
DEVICE = torch.device('cpu' if FORCE_CPU or not torch.cuda.is_available() else 'cuda')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'Compute capability: {torch.cuda.get_device_capability(0)}')
print(f'PyTorch: {torch.__version__}')
print(f'timm   : {timm.__version__}')

# Fail-fast GPU/PyTorch kernel compatibility check.
if DEVICE.type == 'cuda':
    try:
        _probe = torch.nn.Conv2d(3, 8, kernel_size=3, padding=1).to(DEVICE)
        _dummy = torch.randn(1, 3, 32, 32, device=DEVICE)
        with torch.no_grad():
            _ = _probe(_dummy)
        del _probe, _dummy
        torch.cuda.empty_cache()
        print('GPU compatibility check: OK')
    except Exception as e:
        raise RuntimeError(
            'GPU compatibility check FAILED for the assigned GPU '
            f'({torch.cuda.get_device_name(0)}, capability {torch.cuda.get_device_capability(0)}). '
            'Go to Notebook Settings -> Accelerator and select "GPU T4 x2", then re-run. '
            f'Original error: {e}'
        ) from e

GLOBAL_START_TIME = time.time()
print(f'\nGlobal wall-clock timer started.')

## 2. Config

In [ ]:
@dataclass
class CFG:
    # Paths
    DATA_DIR:              str   = '/kaggle/input/datasets/maheshwarmishra/freuid-data'
    WEIGHTS_PATH:          str   = ('/kaggle/input/models/timm/tf-efficientnet/'
                                    'pytorch/tf-efficientnet-b4/1/'
                                    'tf_efficientnet_b4_aa-818f208c.pth')
    OUTPUT_DIR:            str   = '/kaggle/working'
    CHECKPOINT_DIR_V4:     str   = '/kaggle/working/checkpoints_v4_source'   # warm-start source, never overwritten
    CHECKPOINT_DIR_V5:     str   = '/kaggle/working/checkpoints_v8'          # fine-tune output
    USE_FULL_DATA:         bool  = True
    TRAIN_LABELS_FILE:     str   = ''
    TEST_IMG_SUBDIR:       str   = 'public_test'

    # Model (must match v4 exactly for state_dict compatibility)
    BACKBONE:              str   = 'tf_efficientnet_b4'
    PRETRAINED:            bool  = True
    IMG_SIZE:              int   = 320
    DROP_RATE:             float = 0.3
    USE_METADATA:          bool  = True
    N_DOC_TYPES:           int   = 256   # recomputed in Section 5
    DOC_EMB_DIM:           int   = 16

    # Fine-tune training
    N_FOLDS:               int   = 5
    FT_EPOCHS:              int   = 6      # small, time-boxed budget -- NOT v4's EPOCHS=20
    FT_WARMUP_EPOCHS:       int   = 1
    BATCH_SIZE:             int   = 16
    GRAD_ACCUM:              int   = 2
    NUM_WORKERS:             int   = 0
    PIN_MEMORY:              bool  = True
    VAL_SAMPLE_SIZE:         int   = 5000

    # Fine-tune optimizer -- lower LR than v4's full-training LR (2e-4), to nudge the
    # decision boundary away from the shortcut without destroying learned features
    FT_LR:                  float = 5e-5
    FT_LR_MIN:               float = 1e-6
    WEIGHT_DECAY:            float = 1e-4
    GRAD_CLIP:                float = 1.0

    # Loss (identical to v4)
    LOSS_TYPE:               str   = 'pauc_focal'
    FOCAL_GAMMA:              float = 2.0
    FOCAL_ALPHA:              float = 0.75
    FPR_MAX:                  float = 0.10
    PAUC_WEIGHT:               float = 0.5

    # Augmentation (identical probabilities to v4's full production run)
    AUG_P_JPEG:                float = 0.5
    AUG_P_NOISE:                 float = 0.3
    AUG_P_BLUR:                   float = 0.3
    AUG_P_MOIRE:                   float = 0.2
    AUG_P_WARP:                     float = 0.15

    # Face occlusion (v4's fix, kept -- it worked, no reason to remove it)
    FACE_OCCLUSION_P:                 float = 0.3
    FACE_OCCLUSION_PAD:                 float = 0.2
    FACE_DETECT_MAX_SIDE:                 int   = 640

    # NEW (v5): noise/sharpness equalization -- targets calibrated from the Option-A
    # diagnostic's NATIVE-resolution stats (MAURITIUS/ID noise_sigma~0.6-0.65,
    # laplacian_var~256-273; DL-format noise_sigma~1.4-1.42, laplacian_var~1081-1101,
    # with Mozambique reaching noise~3.0, laplacian_var~5000). Mid-band targets pull BOTH
    # ends toward a common middle -- applied pre-resize, same insertion point as face
    # occlusion, so these native-resolution-calibrated bands stay valid.
    NOISE_EQ_P:                         float = 0.95
    NOISE_EQ_RANGE:                       Tuple[float, float] = (0.9, 1.6)
    SHARP_EQ_RANGE:                         Tuple[float, float] = (500.0, 900.0)

    # Early stopping (soft signal only -- every epoch is saved regardless, see Section 14)
    FT_PATIENCE:                       int   = 3
    FT_MIN_DELTA:                       float = 0.0005

    # Safety
    MAX_WALLCLOCK_HOURS:                 float = 5.5

    # Calibration
    CALIB_METHOD:                         str   = 'temperature'
    AMP:                                    bool  = True

    def __post_init__(self):
        if not self.TRAIN_LABELS_FILE:
            self.TRAIN_LABELS_FILE = (
                'train_labels.csv' if self.USE_FULL_DATA
                else 'train_sample_labels.csv'
            )


CFG = CFG()
Path(CFG.CHECKPOINT_DIR_V4).mkdir(parents=True, exist_ok=True)
Path(CFG.CHECKPOINT_DIR_V5).mkdir(parents=True, exist_ok=True)
Path(CFG.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print('Config loaded.')
print(f'  FT_EPOCHS            : {CFG.FT_EPOCHS}  (small, time-boxed -- not a full retrain)')
print(f'  FT_LR                : {CFG.FT_LR}')
print(f'  NOISE_EQ_P           : {CFG.NOISE_EQ_P}')
print(f'  NOISE_EQ_RANGE       : {CFG.NOISE_EQ_RANGE}')
print(f'  SHARP_EQ_RANGE       : {CFG.SHARP_EQ_RANGE}')
print(f'  MAX_WALLCLOCK_HOURS  : {CFG.MAX_WALLCLOCK_HOURS}')

## 3. Restore v4 Checkpoints (warm-start source — never overwritten)

Edit `CKPT_V4_SOURCE_CANDIDATES` to match your attached v4-output dataset. These are copied into `CHECKPOINT_DIR_V4` and treated as read-only for the rest of this notebook — v5 output always goes to a separate directory, so your current 0.22-scoring submission stays reproducible no matter what happens below.

In [ ]:
CKPT_V4_SOURCE_CANDIDATES = [
    '/kaggle/input/datasets/maheshwarmishra/v4fold0',
]

existing_v4 = list(Path(CFG.CHECKPOINT_DIR_V4).glob('fold*_best.pth'))
if existing_v4:
    print(f'CHECKPOINT_DIR_V4 already populated: {[p.name for p in existing_v4]}')
else:
    copied = []
    for src_dir in CKPT_V4_SOURCE_CANDIDATES:
        src_path = Path(src_dir)
        if not src_path.exists():
            continue
        for ckpt_file in src_path.glob('fold*_best.pth'):
            dest = Path(CFG.CHECKPOINT_DIR_V4) / ckpt_file.name
            shutil.copy2(ckpt_file, dest)
            copied.append(ckpt_file.name)
    print(f'v4 checkpoints copied into {CFG.CHECKPOINT_DIR_V4}: {copied}')

v4_ckpts_found = sorted(Path(CFG.CHECKPOINT_DIR_V4).glob('fold*_best.pth'))
if not v4_ckpts_found:
    raise RuntimeError(
        'No v4 checkpoints found. This notebook fine-tunes FROM v4 weights -- it cannot proceed '
        'without them. Check the Input sidebar for your attached v4-output dataset and update '
        'CKPT_V4_SOURCE_CANDIDATES above.'
    )
print(f'\nv4 warm-start checkpoints available: {[p.name for p in v4_ckpts_found]}')

# Any existing in-progress v5 checkpoints (resuming an interrupted fine-tune session)
existing_v5 = list(Path(CFG.CHECKPOINT_DIR_V5).glob('fold*_v5_latest.pth'))
if existing_v5:
    print(f'Found in-progress v5 checkpoints -- will resume: {[p.name for p in existing_v5]}')
else:
    print('No in-progress v5 checkpoints -- all folds will fine-tune from v4 weights, epoch 0.')

## 4. Restore Face-Bbox Cache (MANDATORY — do not skip)

Recomputing this cache took ~2.1 hours in v4 (69,352 images through a Haar cascade). This notebook refuses to silently eat that time again. Attach the v4 output dataset containing `face_bbox_cache.pkl` and update `FACE_CACHE_CANDIDATES` below. If you genuinely want to allow a recompute (NOT recommended given the time budget), set `ALLOW_FACE_BBOX_RECOMPUTE = True`.

In [ ]:
FACE_CACHE_CANDIDATES = [
    '/kaggle/input/datasets/maheshwarmishra/v4fold0/face_bbox_cache.pkl',
]
ALLOW_FACE_BBOX_RECOMPUTE = False   # leave False unless you accept a ~2+ hour cost

FACE_BBOX_CACHE_PATH = Path(CFG.OUTPUT_DIR) / 'face_bbox_cache.pkl'

if not FACE_BBOX_CACHE_PATH.exists():
    found = False
    for cand in FACE_CACHE_CANDIDATES:
        cand_path = Path(cand)
        if cand_path.exists():
            shutil.copy2(cand_path, FACE_BBOX_CACHE_PATH)
            print(f'Face bbox cache restored from: {cand_path}')
            found = True
            break
    if not found and not ALLOW_FACE_BBOX_RECOMPUTE:
        raise RuntimeError(
            'face_bbox_cache.pkl not found in any FACE_CACHE_CANDIDATES path, and '
            'ALLOW_FACE_BBOX_RECOMPUTE is False. Recomputing this cache costs ~2.1 hours out of '
            'your ~5-6 hour budget -- attach the v4 output dataset (which contains this file) as '
            'an input, update FACE_CACHE_CANDIDATES above to match its mounted path, and re-run. '
            'If you have deliberately decided to accept the recompute cost, set '
            'ALLOW_FACE_BBOX_RECOMPUTE = True and re-run this cell.'
        )
    elif not found:
        print('WARNING: cache not found, proceeding to recompute -- this will cost ~2+ hours.')
else:
    print(f'Face bbox cache already present at {FACE_BBOX_CACHE_PATH}.')

## 5. Data Loading and Cross-Validation Splits

**Must match v4 exactly** — same `SEED`, same `StratifiedKFold` call, same stratification key — so fold membership lines up with the checkpoints we're warm-starting from.

In [ ]:
IMAGE_EXTENSIONS = ('.jpeg', '.jpg', '.png', '.webp', '.bmp')

def find_images_in_dir(directory: Path) -> List[Path]:
    files = []
    for ext in IMAGE_EXTENSIONS:
        files += list(directory.glob(f'*{ext}'))
    return sorted(files)


def find_images_robust(base_dir: Path, subdir: str) -> Tuple[Path, List[Path]]:
    flat = base_dir / subdir
    files = find_images_in_dir(flat) if flat.exists() else []
    if files:
        return flat, files
    doubled = base_dir / subdir / subdir
    files = find_images_in_dir(doubled) if doubled.exists() else []
    if files:
        return doubled, files
    if flat.exists():
        for child in sorted(flat.iterdir()):
            if child.is_dir():
                files = find_images_in_dir(child)
                if files:
                    return child, files
    return flat, []


def resolve_image_path(base_dir: Path, rel_path: str) -> Path:
    c = base_dir / rel_path
    if c.exists(): return c
    parts = Path(rel_path).parts
    if len(parts) > 1:
        d = base_dir / parts[0] / rel_path
        if d.exists(): return d
    f = base_dir / Path(rel_path).name
    if f.exists(): return f
    return c


data_dir = Path(CFG.DATA_DIR)

labels_path = data_dir / CFG.TRAIN_LABELS_FILE
train_df = pd.read_csv(labels_path)
train_df['is_digital'] = train_df['is_digital'].astype(bool).astype(int)

all_types = train_df['type'].unique()
type2idx  = {t: i + 1 for i, t in enumerate(sorted(all_types))}
type2idx['<UNK>'] = 0
train_df['type_idx'] = train_df['type'].map(type2idx).fillna(0).astype(int)
CFG.N_DOC_TYPES = len(type2idx) + 1
print(f'Recomputed N_DOC_TYPES: {CFG.N_DOC_TYPES} (must match checkpoint embedding size)')

actual_test_dir, test_files = find_images_robust(data_dir, CFG.TEST_IMG_SUBDIR)
rel_prefix = actual_test_dir.relative_to(data_dir)
test_df = pd.DataFrame({
    'id':         [p.stem for p in test_files],
    'image_path': [str(rel_prefix / p.name) for p in test_files],
    'is_digital': 1,   # matches the configuration confirmed on the leaderboard
    'type_idx':   0,
})
print(f'Test images: {len(test_df)}')

sub_csv = data_dir / 'sample_submission.csv'
sub_df  = pd.read_csv(sub_csv) if sub_csv.exists() else pd.DataFrame()
if sub_df.empty:
    raise RuntimeError(f'sample_submission.csv not found at {sub_csv}')

fraud_rate = float(train_df['label'].mean())
print(f'fraud_rate (fill_value, held constant): {fraud_rate:.4f}')

# Cross-validation -- IDENTICAL logic/seed to v4, so fold membership matches the checkpoints.
train_df['strat_key'] = (
    train_df['label'].astype(str) + '_' +
    train_df['is_digital'].astype(str) + '_' +
    train_df['type'].astype(str)
)
min_stratum  = train_df['strat_key'].value_counts().min()
n_folds_safe = max(2, min(CFG.N_FOLDS, min_stratum))

skf = StratifiedKFold(n_splits=n_folds_safe, shuffle=True, random_state=SEED)
train_df['fold'] = -1
for fold_idx, (_, val_idx) in enumerate(skf.split(train_df, train_df['strat_key'])):
    train_df.loc[train_df.index[val_idx], 'fold'] = fold_idx

assert (train_df['fold'] == -1).sum() == 0
print(f'Folds created: {n_folds_safe} (must match v4 -- verify against v4 logs if unsure)')
print(train_df.groupby('fold')['label'].value_counts().unstack())

## 6. FREUID Metric (identical to v4)

In [ ]:
def compute_det_curve(labels: np.ndarray, scores: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    thresholds = np.unique(scores)[::-1]
    n_genuine  = (labels == 0).sum()
    n_attack   = (labels == 1).sum()
    assert n_genuine > 0 and n_attack > 0
    far_list, frr_list = [], []
    for thresh in thresholds:
        preds = (scores >= thresh).astype(int)
        FP = ((preds == 1) & (labels == 0)).sum()
        FN = ((preds == 0) & (labels == 1)).sum()
        TP = ((preds == 1) & (labels == 1)).sum()
        TN = ((preds == 0) & (labels == 0)).sum()
        far_list.append(FP / (FP + TN) if (FP + TN) > 0 else 0.0)
        frr_list.append(FN / (FN + TP) if (FN + TP) > 0 else 0.0)
    return thresholds, np.array(far_list), np.array(frr_list)


def compute_audet(far: np.ndarray, frr: np.ndarray) -> float:
    order = np.argsort(far)
    far_s = np.concatenate([[0.0], far[order], [1.0]])
    frr_s = np.concatenate([[1.0], frr[order], [0.0]])
    trapz = getattr(np, 'trapezoid', None) or np.trapz
    return float(trapz(frr_s, far_s))


def compute_apcer_at_bpcer(labels: np.ndarray, scores: np.ndarray, bpcer_target: float = 0.01) -> float:
    _, far, frr = compute_det_curve(labels, scores)
    valid = far <= bpcer_target
    return float(frr[valid].min()) if valid.any() else 1.0


def compute_freuid_score(labels: np.ndarray, scores: np.ndarray, bpcer_target: float = 0.01, eps: float = 1e-8) -> Dict[str, float]:
    labels = np.asarray(labels)
    scores = np.asarray(scores)
    _, far, frr = compute_det_curve(labels, scores)
    audet      = compute_audet(far, frr)
    apcer_1pct = compute_apcer_at_bpcer(labels, scores, bpcer_target)
    g_audet = 1.0 - audet
    g_apcer = 1.0 - apcer_1pct
    denom   = g_audet + g_apcer
    freuid  = (1.0 - (2.0 * g_audet * g_apcer) / denom if abs(denom) > eps else 1.0)
    try:
        auc_roc = roc_auc_score(labels, scores)
    except Exception:
        auc_roc = float('nan')
    return {'freuid': freuid, 'audet': audet, 'apcer_1pct': apcer_1pct,
            'g_audet': g_audet, 'g_apcer': g_apcer, 'auc_roc': auc_roc}


def evaluate_breakdown(df: pd.DataFrame, scores: np.ndarray, bpcer_target: float = 0.01) -> pd.DataFrame:
    df = df.reset_index(drop=True)
    rows = []
    m = compute_freuid_score(df['label'].values, scores, bpcer_target)
    rows.append({'group': 'OVERALL', 'subset': 'all', 'n': len(df), **m})
    for doc_type, grp in df.groupby('type'):
        if len(grp) < 20 or len(grp['label'].unique()) < 2:
            continue
        m = compute_freuid_score(grp['label'].values, scores[grp.index.values], bpcer_target)
        rows.append({'group': 'doc_type', 'subset': doc_type, 'n': len(grp), **m})
    return pd.DataFrame(rows)

print('FREUID metric functions defined.')

## 7. Augmentation Pipeline

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


def build_recapture_aug() -> A.Compose:
    return A.Compose([
        A.OneOf([
            A.MotionBlur(blur_limit=(3, 7), p=1.0),
            A.GaussianBlur(blur_limit=(3, 5), p=1.0),
            A.Defocus(radius=(1, 3), p=1.0),
        ], p=CFG.AUG_P_BLUR),
        A.OneOf([
            A.GaussNoise(var_limit=(5, 30), p=1.0),
            A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=1.0),
        ], p=CFG.AUG_P_NOISE),
        A.ImageCompression(quality_lower=40, quality_upper=85, p=CFG.AUG_P_JPEG),
        A.Perspective(scale=(0.02, 0.08), p=CFG.AUG_P_WARP),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.4),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=0.3),
        A.GridDistortion(num_steps=4, distort_limit=0.08, p=CFG.AUG_P_MOIRE),
    ], p=1.0)


def build_train_transform(img_size: int = CFG.IMG_SIZE) -> A.Compose:
    return A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, border_mode=0, p=0.5),
        A.CoarseDropout(max_holes=4, max_height=img_size // 8, max_width=img_size // 8, fill_value=0, p=0.3),
        build_recapture_aug(),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


def build_val_transform(img_size: int = CFG.IMG_SIZE) -> A.Compose:
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


print(f'Transforms defined. IMG_SIZE={CFG.IMG_SIZE}')

### 7a. Face Occlusion (v4's fix, kept unchanged) + Noise/Sharpness Equalization (NEW, v5)

Both are train-only callables applied to the raw, native-resolution image inside `FREUIDDataset.__getitem__`, BEFORE the albumentations `Compose` (resize/other augs/normalize) runs. Face occlusion first (needs original-image-space bbox coordinates), then equalization (target bands were calibrated from native-resolution diagnostic stats, so this must also run pre-resize to stay valid).

In [ ]:
_face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

def detect_face_bbox(image_path: Path, max_side: int = CFG.FACE_DETECT_MAX_SIDE):
    try:
        img = cv2.imread(str(image_path))
        if img is None:
            return None
        h0, w0 = img.shape[:2]
        scale = max_side / max(h0, w0) if max(h0, w0) > max_side else 1.0
        small = cv2.resize(img, (int(w0 * scale), int(h0 * scale))) if scale != 1.0 else img
        gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
        faces = _face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=4)
        if len(faces) == 0:
            return None
        x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
        inv = 1.0 / scale
        return (int(x * inv), int(y * inv), int(w * inv), int(h * inv))
    except Exception:
        return None


def build_face_bbox_cache(df: pd.DataFrame, cache_path: Path = FACE_BBOX_CACHE_PATH) -> Dict:
    if cache_path.exists():
        with open(cache_path, 'rb') as f:
            cache = pickle.load(f)
        missing = set(df['id']) - set(cache.keys())
        if not missing:
            print(f'Loaded existing face bbox cache: {len(cache)} entries (all ids covered).')
            return cache
        print(f'Cache missing {len(missing)} ids -- computing those only.')
    else:
        cache = {}
        missing = set(df['id'])

    t0 = time.time()
    subset = df[df['id'].isin(missing)]
    for i, (_, row) in enumerate(subset.iterrows()):
        p = resolve_image_path(data_dir, row['image_path'])
        cache[row['id']] = detect_face_bbox(p)
        if (i + 1) % 5000 == 0:
            print(f'  {i+1}/{len(subset)} processed, {time.time()-t0:.0f}s elapsed')

    with open(cache_path, 'wb') as f:
        pickle.dump(cache, f)
    print(f'Face bbox cache updated in {time.time()-t0:.0f}s.')
    return cache


FACE_BBOX_CACHE = build_face_bbox_cache(train_df)


class FaceOcclusion:
    """Unchanged from v4 -- this fix worked (LB 0.279 -> 0.221), no reason to touch it."""

    def __init__(self, bbox_cache: Dict, p: float = CFG.FACE_OCCLUSION_P, pad: float = CFG.FACE_OCCLUSION_PAD):
        self.bbox_cache = bbox_cache
        self.p = p
        self.pad = pad

    def __call__(self, image: np.ndarray, image_id: str) -> np.ndarray:
        if np.random.random() > self.p:
            return image
        bbox = self.bbox_cache.get(image_id)
        if bbox is None:
            return image
        x, y, w, h = bbox
        pad_w, pad_h = int(w * self.pad), int(h * self.pad)
        H, W = image.shape[:2]
        x0, y0 = max(0, x - pad_w), max(0, y - pad_h)
        x1, y1 = min(W, x + w + pad_w), min(H, y + h + pad_h)
        if x1 <= x0 or y1 <= y0:
            return image
        image = image.copy()
        if np.random.random() > 0.5:
            image[y0:y1, x0:x1] = 0
        else:
            region = image[y0:y1, x0:x1]
            if region.size > 0:
                image[y0:y1, x0:x1] = cv2.GaussianBlur(region, (15, 15), 0)
        return image


class NoiseSharpnessEqualizer:
    """NEW (v5). Measures noise_sigma (Immerkaer's estimator) and laplacian_var (sharpness) on the
    raw image, then nudges both toward a randomly-sampled mid-band target -- adding noise / an
    unsharp-mask boost to low-noise-low-sharpness images (MAURITIUS/ID's signature), and applying
    blur to high-noise-high-sharpness images (the DL-format types' signature, esp. Mozambique's
    extreme outliers). The goal is to make noise/sharpness statistics uninformative about document
    format by training time, so the model can't use them as a type-correlated shortcut."""

    _KERNEL = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], dtype=np.float64)

    def __init__(self, noise_range=CFG.NOISE_EQ_RANGE, sharp_range=CFG.SHARP_EQ_RANGE, p=CFG.NOISE_EQ_P):
        self.noise_range = noise_range
        self.sharp_range = sharp_range
        self.p = p

    def _measure_noise(self, gray: np.ndarray) -> float:
        H, W = gray.shape
        if H < 3 or W < 3:
            return 0.0
        conv = cv2.filter2D(gray, -1, self._KERNEL)
        return float(np.sqrt(np.pi / 2) * np.sum(np.abs(conv)) / (6 * (W - 2) * (H - 2)))

    def _measure_sharp(self, gray: np.ndarray) -> float:
        return float(cv2.Laplacian(gray, cv2.CV_64F).var())

    def __call__(self, image: np.ndarray) -> np.ndarray:
        if np.random.random() > self.p:
            return image

        out = image.astype(np.float32)
        target_sharp = float(np.random.uniform(*self.sharp_range))

        # Sharpness equalization -- iterate a few cheap blur passes for extreme outliers
        # (some DL-format images run far above target; a single small-kernel blur is not
        # always enough).
        for _ in range(3):
            gray = cv2.cvtColor(np.clip(out, 0, 255).astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float64)
            cur_sharp = self._measure_sharp(gray)
            if cur_sharp <= target_sharp * 1.15:
                break
            out = cv2.GaussianBlur(out, (5, 5), 0)

        gray = cv2.cvtColor(np.clip(out, 0, 255).astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float64)
        cur_sharp = self._measure_sharp(gray)
        if cur_sharp < target_sharp * 0.6:
            blurred = cv2.GaussianBlur(out, (3, 3), 0)
            out = out + 0.6 * (out - blurred)

        # Noise equalization, measured after any blur/sharpen above.
        gray = cv2.cvtColor(np.clip(out, 0, 255).astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float64)
        cur_noise = self._measure_noise(gray)
        target_noise = float(np.random.uniform(*self.noise_range))
        if cur_noise < target_noise:
            extra_sigma = max(0.0, target_noise - cur_noise)
            noise = np.random.normal(0, extra_sigma, out.shape).astype(np.float32)
            out = out + noise

        return np.clip(out, 0, 255).astype(np.uint8)


print('FaceOcclusion (v4) and NoiseSharpnessEqualizer (v5, NEW) defined.')

## 8. Dataset Class and Weighted Sampler

In [ ]:
class FREUIDDataset(Dataset):
    def __init__(self, df: pd.DataFrame, data_dir: Path,
                 transform: Optional[A.Compose] = None,
                 is_train: bool = True,
                 face_occluder: Optional['FaceOcclusion'] = None,
                 noise_equalizer: Optional['NoiseSharpnessEqualizer'] = None) -> None:
        self.df              = df.reset_index(drop=True)
        self.data_dir        = data_dir
        self.transform        = transform
        self.is_train          = is_train
        self.face_occluder      = face_occluder if is_train else None
        self.noise_equalizer      = noise_equalizer if is_train else None

    def __len__(self) -> int:
        return len(self.df)

    def _resolve(self, rel_path: str) -> Path:
        c = self.data_dir / rel_path
        if c.exists(): return c
        parts = Path(rel_path).parts
        if len(parts) > 1:
            d = self.data_dir / parts[0] / rel_path
            if d.exists(): return d
        f = self.data_dir / Path(rel_path).name
        if f.exists(): return f
        return c

    def __getitem__(self, idx: int) -> Dict:
        row = self.df.iloc[idx]
        img_path = self._resolve(row['image_path'])
        try:
            image = np.array(Image.open(img_path).convert('RGB'))
        except Exception as e:
            print(f'Warning: cannot load {img_path}: {e}')
            image = np.zeros((CFG.IMG_SIZE, CFG.IMG_SIZE, 3), dtype=np.uint8)

        # Native-resolution preprocessing, in order: face occlusion, then noise/sharpness
        # equalization. Both are no-ops (probability-gated) at val/inference time.
        if self.face_occluder is not None:
            image = self.face_occluder(image, row['id'])
        if self.noise_equalizer is not None:
            image = self.noise_equalizer(image)

        if self.transform is not None:
            image = self.transform(image=image)['image']

        out = {
            'image':      image,
            'is_digital': torch.tensor(float(row.get('is_digital', 0)), dtype=torch.float32),
            'type_idx':   torch.tensor(int(row.get('type_idx', 0)), dtype=torch.long),
            'id':         str(row['id']),
        }
        if self.is_train:
            out['label'] = torch.tensor(int(row['label']), dtype=torch.long)
        return out


def build_weighted_sampler(df: pd.DataFrame) -> WeightedRandomSampler:
    label_w = df['label'].map((1.0 / df['label'].value_counts()).to_dict()).values.astype(float)
    strat_w = (1.0 / df.groupby(['type', 'is_digital'])['label'].transform('count').values.astype(float))
    weights = torch.tensor(label_w * strat_w, dtype=torch.double)
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

print('FREUIDDataset and build_weighted_sampler defined.')

## 9. Model Architecture (identical to v4)

In [ ]:
class FREUIDModel(nn.Module):
    def __init__(
        self,
        backbone_name: str   = CFG.BACKBONE,
        pretrained:    bool  = CFG.PRETRAINED,
        weights_path:  str   = CFG.WEIGHTS_PATH,
        n_doc_types:   int   = CFG.N_DOC_TYPES,
        doc_emb_dim:   int   = CFG.DOC_EMB_DIM,
        drop_rate:     float = CFG.DROP_RATE,
        use_metadata:  bool  = CFG.USE_METADATA,
    ) -> None:
        super().__init__()
        self.use_metadata = use_metadata
        self.backbone = timm.create_model(backbone_name, pretrained=False, num_classes=0, global_pool='avg')

        if pretrained:
            wp = Path(weights_path)
            if wp.exists():
                state_dict = torch.load(wp, map_location='cpu', weights_only=False)
                self.backbone.load_state_dict(state_dict, strict=False)

        feat_dim = self.backbone.num_features
        if use_metadata:
            self.doc_embedding = nn.Embedding(num_embeddings=n_doc_types, embedding_dim=doc_emb_dim, padding_idx=0)
            meta_dim = doc_emb_dim + 1
        else:
            meta_dim = 0

        in_dim = feat_dim + meta_dim
        self.head = nn.Sequential(
            nn.LayerNorm(in_dim), nn.Dropout(drop_rate),
            nn.Linear(in_dim, 256), nn.GELU(),
            nn.Dropout(drop_rate / 2), nn.Linear(256, 1),
        )

    def forward(self, image: torch.Tensor, is_digital: torch.Tensor, type_idx: torch.Tensor) -> torch.Tensor:
        feats = self.backbone(image)
        if self.use_metadata:
            feats = torch.cat([feats, self.doc_embedding(type_idx), is_digital.unsqueeze(1)], dim=1)
        return self.head(feats).squeeze(1)

print('FREUIDModel defined.')

## 10. Loss Functions (identical to v4)

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=CFG.FOCAL_ALPHA, gamma=CFG.FOCAL_GAMMA):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        targets = targets.float()
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probs = torch.sigmoid(logits)
        p_t = probs * targets + (1 - probs) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (alpha_t * (1 - p_t) ** self.gamma * bce).mean()


class PartialAUCLoss(nn.Module):
    def __init__(self, fpr_max=CFG.FPR_MAX, margin=1.0):
        super().__init__()
        self.fpr_max = fpr_max
        self.margin = margin

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs = torch.sigmoid(logits)
        targets = targets.float()
        attack = probs[targets == 1]
        genuine = probs[targets == 0]
        if len(attack) == 0 or len(genuine) == 0:
            return torch.tensor(0.0, device=logits.device, requires_grad=True)
        k = max(1, int(math.ceil(self.fpr_max * len(genuine))))
        hard_genuine, _ = torch.topk(genuine, k=k)
        diff = attack.unsqueeze(1) - hard_genuine.unsqueeze(0)
        return F.relu(self.margin - diff).mean()


class CombinedLoss(nn.Module):
    def __init__(self, pauc_weight=CFG.PAUC_WEIGHT, fpr_max=CFG.FPR_MAX):
        super().__init__()
        self.focal = FocalLoss()
        self.pauc = PartialAUCLoss(fpr_max=fpr_max)
        self.w = pauc_weight

    def forward(self, logits, targets):
        focal_loss = self.focal(logits, targets)
        pauc_loss = self.pauc(logits, targets)
        total = (1 - self.w) * focal_loss + self.w * pauc_loss
        return total, {'focal_loss': focal_loss.item(), 'pauc_loss': pauc_loss.item()}


def get_loss_fn() -> nn.Module:
    if CFG.LOSS_TYPE == 'focal':      return FocalLoss()
    if CFG.LOSS_TYPE == 'pauc_focal': return CombinedLoss()
    if CFG.LOSS_TYPE == 'bce':        return nn.BCEWithLogitsLoss()
    raise ValueError(f'Unknown loss type: {CFG.LOSS_TYPE}')

print('Loss functions defined.')

## 11. Score Calibration (identical to v4)

In [ ]:
class TemperatureScaler:
    def __init__(self): self.temperature = 1.0

    def fit(self, logits: np.ndarray, labels: np.ndarray) -> 'TemperatureScaler':
        def nll(t):
            t = float(t[0])
            if t <= 0: return 1e9
            p = np.clip(1.0 / (1.0 + np.exp(-logits / t)), 1e-7, 1 - 1e-7)
            return -np.mean(labels * np.log(p) + (1 - labels) * np.log(1 - p))
        res = scipy.optimize.minimize(nll, [1.0], method='L-BFGS-B', bounds=[(0.05, 20.0)])
        self.temperature = float(res.x[0])
        return self

    def transform(self, logits: np.ndarray) -> np.ndarray:
        return (1.0 / (1.0 + np.exp(-logits / self.temperature))).astype(np.float32)


class IsotonicCalibrator:
    def __init__(self): self.iso = IsotonicRegression(out_of_bounds='clip')

    def fit(self, logits: np.ndarray, labels: np.ndarray) -> 'IsotonicCalibrator':
        scores = 1.0 / (1.0 + np.exp(-logits))
        self.iso.fit(scores, labels)
        return self

    def transform(self, logits: np.ndarray) -> np.ndarray:
        scores = 1.0 / (1.0 + np.exp(-logits))
        return self.iso.predict(scores).astype(np.float32)


def get_calibrator():
    if CFG.CALIB_METHOD == 'temperature': return TemperatureScaler()
    if CFG.CALIB_METHOD == 'isotonic':    return IsotonicCalibrator()
    raise ValueError(f'Unknown calibration method: {CFG.CALIB_METHOD}')

print('Calibration classes defined.')

## 12. Inference and Submission

In [ ]:
TTA_TRANSFORMS = [
    build_val_transform(),
    A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
        A.HorizontalFlip(p=1.0),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ]),
]


@torch.no_grad()
def predict(model: nn.Module, df: pd.DataFrame, calibrator=None, use_tta: bool = True) -> np.ndarray:
    model.eval()
    transforms = TTA_TRANSFORMS if use_tta else [build_val_transform()]
    all_logits = []
    for tfm in transforms:
        ds = FREUIDDataset(df, data_dir, tfm, is_train=False)
        loader = DataLoader(ds, batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
                             num_workers=CFG.NUM_WORKERS, pin_memory=CFG.PIN_MEMORY)
        tfm_logits = []
        for batch in loader:
            with torch.cuda.amp.autocast(enabled=CFG.AMP):
                out = model(batch['image'].to(DEVICE), batch['is_digital'].to(DEVICE), batch['type_idx'].to(DEVICE))
            tfm_logits.append(out.cpu().float().numpy())
        all_logits.append(np.concatenate(tfm_logits))
    mean_logits = np.mean(all_logits, axis=0)
    if calibrator is not None:
        return calibrator.transform(mean_logits)
    return (1.0 / (1.0 + np.exp(-mean_logits))).astype(np.float32)


def rank_average(score_arrays: List[np.ndarray]) -> np.ndarray:
    ranks = [scipy.stats.rankdata(s) / len(s) for s in score_arrays]
    return np.mean(ranks, axis=0).astype(np.float32)


def generate_submission(scores: np.ndarray, test_df_: pd.DataFrame, filename: str, fill_value: float) -> pd.DataFrame:
    assert len(scores) == len(test_df_)
    assert np.all((scores >= 0) & (scores <= 1))
    score_col = [c for c in sub_df.columns if c != 'id'][0]
    score_map = dict(zip(test_df_['id'].astype(str), scores))
    full_sub = sub_df.copy()
    full_sub[score_col] = full_sub['id'].astype(str).map(score_map).fillna(fill_value)
    n_real = full_sub['id'].astype(str).isin(score_map).sum()
    out_path = Path(CFG.OUTPUT_DIR) / filename
    full_sub.to_csv(out_path, index=False)
    print(f'Saved: {out_path} | rows={len(full_sub)} | real={n_real} | fill({fill_value:.4f})={len(full_sub)-n_real}')
    print(f'Real-row scores: min={scores.min():.4f} max={scores.max():.4f} mean={scores.mean():.4f}')
    return full_sub

print('predict(), rank_average(), generate_submission() defined.')

## 13. Training Step Functions (identical to v4)

In [ ]:
def build_scheduler(optimizer, n_epochs: int, steps_per_epoch: int, warmup_epochs: int):
    total_steps = n_epochs * steps_per_epoch
    warmup_steps = warmup_epochs * steps_per_epoch

    def lr_lambda(step: int) -> float:
        if step < warmup_steps:
            return (CFG.FT_LR_MIN / CFG.FT_LR + (1.0 - CFG.FT_LR_MIN / CFG.FT_LR) * step / max(1, warmup_steps))
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return (CFG.FT_LR_MIN / CFG.FT_LR + 0.5 * (1.0 - CFG.FT_LR_MIN / CFG.FT_LR) * (1.0 + math.cos(math.pi * progress)))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def train_one_epoch(model, loader, optimizer, scheduler, loss_fn, scaler, epoch) -> Dict[str, float]:
    model.train()
    total_loss = 0.0
    all_logits, all_labels = [], []
    n_batches = len(loader)
    optimizer.zero_grad()

    for step, batch in enumerate(loader):
        img  = batch['image'].to(DEVICE)
        dig  = batch['is_digital'].to(DEVICE)
        tidx = batch['type_idx'].to(DEVICE)
        lbl  = batch['label'].to(DEVICE)

        with torch.cuda.amp.autocast(enabled=CFG.AMP and scaler is not None):
            logits = model(img, dig, tidx)
            if isinstance(loss_fn, CombinedLoss):
                loss, _ = loss_fn(logits, lbl)
            else:
                loss = loss_fn(logits, lbl.float())
            loss = loss / CFG.GRAD_ACCUM

        if scaler is not None:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        if (step + 1) % CFG.GRAD_ACCUM == 0 or (step + 1) == n_batches:
            if scaler is not None:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
            else:
                nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
                optimizer.step()
            optimizer.zero_grad()
            scheduler.step()

        total_loss += loss.item() * CFG.GRAD_ACCUM
        all_logits.append(logits.detach().cpu().float())
        all_labels.append(lbl.detach().cpu())

    logits_np = torch.cat(all_logits).numpy()
    labels_np = torch.cat(all_labels).numpy()
    scores_np = 1.0 / (1.0 + np.exp(-logits_np))
    try:
        trn_auc = roc_auc_score(labels_np, scores_np)
    except Exception:
        trn_auc = float('nan')
    return {'loss': total_loss / n_batches, 'auc_roc': trn_auc}


@torch.no_grad()
def validate(model, loader, loss_fn) -> Tuple[Dict[str, float], np.ndarray, np.ndarray, np.ndarray]:
    model.eval()
    total_loss = 0.0
    all_logits, all_labels = [], []
    for batch in loader:
        img  = batch['image'].to(DEVICE)
        dig  = batch['is_digital'].to(DEVICE)
        tidx = batch['type_idx'].to(DEVICE)
        lbl  = batch['label'].to(DEVICE)
        with torch.cuda.amp.autocast(enabled=CFG.AMP):
            logits = model(img, dig, tidx)
            if isinstance(loss_fn, CombinedLoss):
                loss, _ = loss_fn(logits, lbl)
            else:
                loss = loss_fn(logits, lbl.float())
        total_loss += loss.item()
        all_logits.append(logits.cpu().float())
        all_labels.append(lbl.cpu())
    logits_np = torch.cat(all_logits).numpy()
    labels_np = torch.cat(all_labels).numpy()
    scores_np = 1.0 / (1.0 + np.exp(-logits_np))
    metrics = compute_freuid_score(labels_np, scores_np)
    metrics['loss'] = total_loss / len(loader)
    return metrics, scores_np, logits_np, labels_np

print('build_scheduler, train_one_epoch, validate defined.')

## 14. Fine-Tune Training Loop (NEW — the core of v5)

Warm-starts from the v4 checkpoint, runs a small, time-boxed number of epochs with the new equalization augmentation, saves **every epoch unconditionally** (not just on improvement), respects the global wall-clock budget, and isolates failures to a single fold via `try/except` so one fold's problem can't take down the run or lose the other fold's progress.

In [ ]:
def finetune_fold(fold: int) -> Dict:
    """Returns a dict with keys: succeeded, notes, model, calibrator, history, final_state.
    Never raises -- all failure modes are caught and reported in the return dict so the caller
    can always fall back to the v4 checkpoint for this fold."""
    result = {'succeeded': False, 'notes': '', 'model': None, 'calibrator': None,
              'history': [], 'final_state': None}

    print(f'\n{"=" * 60}')
    print(f' Fine-tuning Fold {fold} (v5, warm-start)')
    print(f'{"=" * 60}')

    try:
        v4_ckpt_path = Path(CFG.CHECKPOINT_DIR_V4) / f'fold{fold}_best.pth'
        if not v4_ckpt_path.exists():
            result['notes'] = f'v4 source checkpoint not found: {v4_ckpt_path}'
            print(result['notes'])
            return result

        v5_latest_path = Path(CFG.CHECKPOINT_DIR_V5) / f'fold{fold}_v5_latest.pth'
        v5_best_path   = Path(CFG.CHECKPOINT_DIR_V5) / f'fold{fold}_v5_best.pth'

        trn_df = train_df[train_df['fold'] != fold].reset_index(drop=True)
        val_df = train_df[train_df['fold'] == fold].reset_index(drop=True)
        print(f'Train: {len(trn_df)} | Val: {len(val_df)}')

        face_occluder   = FaceOcclusion(bbox_cache=FACE_BBOX_CACHE, p=CFG.FACE_OCCLUSION_P)
        noise_equalizer = NoiseSharpnessEqualizer(p=CFG.NOISE_EQ_P)
        trn_ds = FREUIDDataset(trn_df, data_dir, build_train_transform(), is_train=True,
                                face_occluder=face_occluder, noise_equalizer=noise_equalizer)

        val_fast_df = val_df.sample(n=min(CFG.VAL_SAMPLE_SIZE, len(val_df)), random_state=SEED).reset_index(drop=True)
        val_ds_fast = FREUIDDataset(val_fast_df, data_dir, build_val_transform(), is_train=True)
        val_ds_full = FREUIDDataset(val_df, data_dir, build_val_transform(), is_train=True)

        sampler = build_weighted_sampler(trn_df)
        trn_loader = DataLoader(trn_ds, batch_size=CFG.BATCH_SIZE, sampler=sampler,
                                 num_workers=CFG.NUM_WORKERS, pin_memory=CFG.PIN_MEMORY, drop_last=True)
        val_loader_fast = DataLoader(val_ds_fast, batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
                                      num_workers=CFG.NUM_WORKERS, pin_memory=CFG.PIN_MEMORY)
        val_loader_full = DataLoader(val_ds_full, batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
                                      num_workers=CFG.NUM_WORKERS, pin_memory=CFG.PIN_MEMORY)

        model = FREUIDModel().to(DEVICE)
        loss_fn = get_loss_fn()

        resume_epoch = 0
        best_ft_freuid = float('inf')

        if v5_latest_path.exists():
            print(f'Resuming in-progress v5 fine-tune: {v5_latest_path}')
            ckpt = torch.load(v5_latest_path, map_location=DEVICE, weights_only=False)
            model.load_state_dict(ckpt['model_state'])
            resume_epoch = ckpt['ft_epoch'] + 1
            best_ft_freuid = ckpt.get('best_ft_freuid', float('inf'))
            print(f'  Resuming from fine-tune epoch {resume_epoch}')
        else:
            print(f'Warm-starting from v4 checkpoint: {v4_ckpt_path}')
            v4_ckpt = torch.load(v4_ckpt_path, map_location=DEVICE, weights_only=False)
            model.load_state_dict(v4_ckpt['model_state'])
            print(f'  v4 stored FREUID was {v4_ckpt["val_metrics"]["freuid"]:.4f} at its epoch {v4_ckpt["epoch"]}')

        param_groups = [
            {'params': model.backbone.parameters(), 'lr': CFG.FT_LR * 0.1},
            {'params': model.head.parameters(),     'lr': CFG.FT_LR},
        ]
        if CFG.USE_METADATA:
            param_groups.append({'params': model.doc_embedding.parameters(), 'lr': CFG.FT_LR})

        optimizer = torch.optim.AdamW(param_groups, weight_decay=CFG.WEIGHT_DECAY)
        scheduler = build_scheduler(optimizer, CFG.FT_EPOCHS, len(trn_loader), CFG.FT_WARMUP_EPOCHS)
        scaler = torch.cuda.amp.GradScaler() if CFG.AMP and torch.cuda.is_available() else None

        history = []
        epochs_since_best = 0
        latest_state = None

        for epoch in range(resume_epoch, CFG.FT_EPOCHS):
            elapsed_hours = (time.time() - GLOBAL_START_TIME) / 3600.0
            if elapsed_hours > CFG.MAX_WALLCLOCK_HOURS:
                print(f'  Wall-clock budget ({CFG.MAX_WALLCLOCK_HOURS}h) reached at epoch {epoch} -- '
                      f'stopping this fold gracefully.')
                result['notes'] = f'wall-clock cutoff at epoch {epoch}'
                break

            try:
                t0 = time.time()
                trn_m = train_one_epoch(model, trn_loader, optimizer, scheduler, loss_fn, scaler, epoch)
                val_m, val_scores, val_logits, val_labels = validate(model, val_loader_fast, loss_fn)
                elapsed = time.time() - t0

                history.append({
                    'epoch': epoch, 'trn_loss': trn_m['loss'], 'trn_auc': trn_m['auc_roc'],
                    'val_freuid': val_m['freuid'], 'val_audet': val_m['audet'],
                    'val_apcer_1pct': val_m['apcer_1pct'], 'val_auc': val_m['auc_roc'],
                    'elapsed_s': elapsed,
                })
                print(f'  Ep {epoch:02d} | trn_loss={trn_m["loss"]:.4f} trn_auc={trn_m["auc_roc"]:.4f} | '
                      f'val_freuid={val_m["freuid"]:.4f} (audet={val_m["audet"]:.4f}, '
                      f'apcer@1%={val_m["apcer_1pct"]:.4f}) | val_auc={val_m["auc_roc"]:.4f} | t={elapsed:.0f}s')

                # Save EVERY epoch unconditionally -- see notebook header for why "only save on
                # new best" would likely never fire given how close to 0 the old best already is.
                latest_state = {
                    'model_state': model.state_dict(), 'val_scores': val_scores,
                    'val_logits': val_logits, 'val_labels': val_labels, 'val_metrics': val_m,
                    'ft_epoch': epoch, 'fold': fold, 'best_ft_freuid': best_ft_freuid,
                }
                torch.save(latest_state, v5_latest_path)

                if val_m['freuid'] < best_ft_freuid - CFG.FT_MIN_DELTA:
                    best_ft_freuid = val_m['freuid']
                    epochs_since_best = 0
                    torch.save(latest_state, v5_best_path)
                    print(f'    New best-within-finetune FREUID={best_ft_freuid:.4f} saved.')
                else:
                    epochs_since_best += 1
                    print(f'    No improvement for {epochs_since_best}/{CFG.FT_PATIENCE} epochs '
                          f'(soft signal -- latest checkpoint saved regardless).')
                    if epochs_since_best >= CFG.FT_PATIENCE:
                        print(f'    Early stopping fold {fold} at epoch {epoch} (soft stop).')
                        result['notes'] = f'early stopped at epoch {epoch}'
                        break

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            except Exception as e:
                print(f'  EXCEPTION during epoch {epoch}: {e}')
                traceback.print_exc()
                result['notes'] = f'exception at epoch {epoch}: {e}'
                break
        else:
            result['notes'] = 'completed all FT_EPOCHS normally'

        if latest_state is None:
            result['notes'] = result['notes'] or 'no epoch completed successfully'
            print(f'No successful epoch for fold {fold} -- caller will fall back to v4 checkpoint.')
            return result

        # Final full validation on whatever was actually saved to disk (defensive reload).
        print('\nRunning final full validation on the saved v5 latest checkpoint...')
        final_ckpt = torch.load(v5_latest_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(final_ckpt['model_state'])
        val_m_full, val_s_full, val_l_full, val_lbl_full = validate(model, val_loader_full, loss_fn)
        print(f'Full val FREUID={val_m_full["freuid"]:.4f} (audet={val_m_full["audet"]:.4f}, '
              f'apcer@1%={val_m_full["apcer_1pct"]:.4f}) | val_auc={val_m_full["auc_roc"]:.4f}')

        final_state = dict(final_ckpt)
        final_state.update({'val_scores': val_s_full, 'val_logits': val_l_full,
                             'val_labels': val_lbl_full, 'val_metrics': val_m_full})
        torch.save(final_state, v5_latest_path)

        calibrator = get_calibrator()
        calibrator.fit(final_state['val_logits'], final_state['val_labels'])

        result.update({'succeeded': True, 'model': model, 'calibrator': calibrator,
                        'history': history, 'final_state': final_state})
        if not result['notes']:
            result['notes'] = 'completed successfully'
        return result

    except Exception as e:
        print(f'EXCEPTION in finetune_fold(fold={fold}) setup: {e}')
        traceback.print_exc()
        result['notes'] = f'setup exception: {e}'
        return result


print('finetune_fold() defined.')

## 15. Safe Model Loading — v5 With Automatic v4 Fallback

Used at submission time: tries the v5 fine-tuned checkpoint first, falls back to the original v4 checkpoint if v5 didn't succeed for that fold. This is what guarantees the EXECUTION cell always produces a valid submission.

In [ ]:
def get_model_and_calibrator_for_fold(fold: int, ft_result: Optional[Dict]) -> Tuple[nn.Module, object, str]:
    """Returns (model, calibrator, source_label). Tries v5 first, falls back to v4."""
    if ft_result is not None and ft_result.get('succeeded'):
        print(f'Fold {fold}: using v5 fine-tuned model ({ft_result["notes"]}).')
        return ft_result['model'], ft_result['calibrator'], 'v5'

    print(f'Fold {fold}: v5 unavailable ({ft_result["notes"] if ft_result else "no result"}) '
          f'-- falling back to v4 checkpoint.')
    v4_ckpt_path = Path(CFG.CHECKPOINT_DIR_V4) / f'fold{fold}_best.pth'
    if not v4_ckpt_path.exists():
        raise RuntimeError(f'Fold {fold}: neither v5 nor v4 checkpoint available. Cannot proceed.')

    ckpt = torch.load(v4_ckpt_path, map_location=DEVICE, weights_only=False)
    model = FREUIDModel().to(DEVICE)
    model.load_state_dict(ckpt['model_state'])
    model.eval()
    calibrator = get_calibrator()
    calibrator.fit(ckpt['val_logits'], ckpt['val_labels'])
    return model, calibrator, 'v4_fallback'

print('get_model_and_calibrator_for_fold() defined.')

## 16. EXECUTION — Fine-Tune All Folds, Compare, Submit

**This is the only cell that runs heavy computation.** Wrapped so that no single fold's failure prevents a valid submission from being generated at the end.

In [ ]:

print('=== Pre-flight ===')
print(f'Train rows        : {len(train_df)}')
print(f'Folds              : {n_folds_safe}')
print(f'FT_EPOCHS per fold : {CFG.FT_EPOCHS}')
print(f'MAX_WALLCLOCK_HOURS: {CFG.MAX_WALLCLOCK_HOURS}')
print(f'v4 source checkpoints available: {[p.name for p in Path(CFG.CHECKPOINT_DIR_V4).glob("fold*_best.pth")]}')
print(f'fill_value (fixed): {fraud_rate:.4f}')
print('=== Starting fine-tune ===\n')

ft_results = {}
for fold in range(n_folds_safe):
    elapsed_hours = (time.time() - GLOBAL_START_TIME) / 3600.0
    if elapsed_hours > CFG.MAX_WALLCLOCK_HOURS:
        print(f'Wall-clock budget already exceeded before fold {fold} could start -- skipping, '
              f'will fall back to v4 for this fold.')
        ft_results[fold] = {'succeeded': False, 'notes': 'skipped -- wall-clock budget exceeded'}
        continue
    try:
        ft_results[fold] = finetune_fold(fold)
    except Exception as e:
        print(f'UNEXPECTED top-level exception for fold {fold}: {e}')
        traceback.print_exc()
        ft_results[fold] = {'succeeded': False, 'notes': f'top-level exception: {e}'}

print('\n' + '=' * 60)
print('FINE-TUNE SUMMARY')
print('=' * 60)
for fold, r in ft_results.items():
    print(f'  fold{fold}: succeeded={r["succeeded"]}  notes="{r["notes"]}"')

# ── v4 vs v5 OOF comparison table ──────────────────────────────────────────
print('\n' + '=' * 60)
print('v4 vs v5 OOF COMPARISON (per fold -- ceiling-effect caveat applies, see header)')
print('=' * 60)
comparison_rows = []
for fold in range(n_folds_safe):
    v4_ckpt_path = Path(CFG.CHECKPOINT_DIR_V4) / f'fold{fold}_best.pth'
    v4_freuid = np.nan
    if v4_ckpt_path.exists():
        v4_ckpt = torch.load(v4_ckpt_path, map_location='cpu', weights_only=False)
        v4_freuid = v4_ckpt['val_metrics']['freuid']
    r = ft_results.get(fold, {})
    v5_freuid = (r['final_state']['val_metrics']['freuid']
                 if r.get('succeeded') else np.nan)
    comparison_rows.append({'fold': fold, 'v4_freuid': v4_freuid, 'v5_freuid': v5_freuid,
                             'v5_status': r.get('notes', 'not run')})
comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.to_string(index=False))

# ── Build ensemble using best-available model per fold (v5, auto-fallback to v4) ──────────
print('\n' + '=' * 60)
print('BUILDING ENSEMBLE (v5 where available, v4 fallback otherwise)')
print('=' * 60)
fold_models = []
source_labels = []
for fold in range(n_folds_safe):
    model, calibrator, source = get_model_and_calibrator_for_fold(fold, ft_results.get(fold))
    fold_models.append((model, calibrator))
    source_labels.append(source)

print(f'\nPer-fold source: {dict(enumerate(source_labels))}')

print('\nRunning ensemble inference on public test set...')
t0 = time.time()
all_scores = [predict(model, test_df, calibrator, use_tta=True) for model, calibrator in fold_models]
ensemble_scores = rank_average(all_scores)
print(f'  done in {time.time()-t0:.0f}s')

sub_v5 = generate_submission(
    ensemble_scores, test_df,
    filename='submission_ensemble_v5.csv',
    fill_value=fraud_rate,   # identical to prior submissions -- LB-isolation discipline maintained
)

total_elapsed_hours = (time.time() - GLOBAL_START_TIME) / 3600.0
print('\n' + '=' * 60)
print('DONE')
print('=' * 60)
print(f'Total wall-clock time: {total_elapsed_hours:.2f}h (budget was {CFG.MAX_WALLCLOCK_HOURS}h)')
print(f'Per-fold source used : {dict(enumerate(source_labels))}')
print(f'Output               : /kaggle/working/submission_ensemble_v5.csv')
print()
if all(s == 'v4_fallback' for s in source_labels):
    print('NOTE: every fold fell back to v4 -- this submission is IDENTICAL in spirit to your')
    print('existing 0.22-scoring submission (same weights). No new signal to gain by submitting')
    print('this one; check the fine-tune summary above to see why v5 did not succeed on any fold.')
else:
    print('At least one fold used the v5 fine-tuned model. This is your last realistic pre-freeze')
    print('opportunity for a genuinely new model variant -- worth spending a submission on it.')
